In [29]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import importlib
import ml_pipeline

In [31]:
train_data_pl = pl.read_csv(r"../data/train.csv",encoding="shift_jis")

test_data_pl =  pl.read_csv(r"../data/test.csv",encoding="shift_jis")

### データ確認（ここから）

In [10]:
train_data_pl.select("species number").unique

<bound method DataFrame.unique of shape: (1_322, 1)
┌────────────────┐
│ species number │
│ ---            │
│ i64            │
╞════════════════╡
│ 1              │
│ 1              │
│ 1              │
│ 1              │
│ 1              │
│ …              │
│ 19             │
│ 19             │
│ 19             │
│ 19             │
│ 19             │
└────────────────┘>

### データ確認（ここまで）

### 各木種類ごとにモデルを作成する：sp=1で確認(ここから)

trainに存在する樹種がtestに存在しないので注意

In [32]:
from ml_pipeline import MoisturePipeline

### 各木種類ごとにモデルを作成するsp=1で確認(ここまで)

#### MLflowを使い各樹ごとモデルのスタッキングを行う

In [33]:
train_df = train_data_pl 
test_df  = test_data_pl

In [35]:
import mlflow
from stacking import (
    train_species_models,
    create_oof,
    train_meta_model,
    StackingModel
)

mlflow.set_experiment("moisture_stacking_2")

with mlflow.start_run():

    # ① speciesモデル
    models, species_list = train_species_models(train_df)

    # ② OOF
    X_meta, y = create_oof(train_df, species_list)

    # ③ metaモデル
    meta_model = train_meta_model(X_meta, y)

    # ④ stacking保存
    mlflow.pyfunc.log_model(
        artifact_path="stacking_model",
        python_model=StackingModel(models, meta_model, species_list)
    )

2026/04/01 22:47:29 INFO mlflow.tracking.fluent: Experiment with name 'moisture_stacking_2' does not exist. Creating a new experiment.
/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/04/01 22:47:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 22:47:30 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


🏃 View run vaunted-ape-769 at: http://mlflow:5000/#/experiments/4/runs/7216da24a61a4524a6c1b4fa98dd97b5
🧪 View experiment at: http://mlflow:5000/#/experiments/4


/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


MlflowException: INVALID_PARAMETER_VALUE: Changing param values is not allowed. Param with key='species' was already logged with value='1' for run ID='7216da24a61a4524a6c1b4fa98dd97b5'. Attempted logging new value '3'.

The cause of this error is typically due to repeated calls
to an individual run_id event logging.

Incorrect Example:
---------------------------------------
with mlflow.start_run():
    mlflow.log_param("depth", 3)
    mlflow.log_param("depth", 5)
---------------------------------------

Which will throw an MlflowException for overwriting a
logged parameter.

Correct Example:
---------------------------------------
with mlflow.start_run():
    with mlflow.start_run(nested=True):
        mlflow.log_param("depth", 3)
    with mlflow.start_run(nested=True):
        mlflow.log_param("depth", 5)
---------------------------------------

Which will create a new nested run for each individual
model and prevent parameter key collisions within the
tracking store.